# Lab 4: Linear Models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Setup plots
%matplotlib inline
sns.set_theme()
np.random.seed(42)

---
## Exercise 1: Simple Linear Regression

In this exercise, we generate a synthetic dataset with a single feature and apply a linear regression model.

### Steps:
1. Generate a synthetic dataset using `make_regression` from `sklearn.datasets`.
2. Split the data into training and test sets.
3. Train a `LinearRegression` model on the training data.
4. Make predictions on the test data.
5. Visualize the results by plotting actual data points and the regression line.
6. Evaluate the model using R² and Mean Squared Error (MSE).

In [ ]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

# 1. Generate synthetic data
X, y = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)

# 2. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the model
# TODO: Create a LinearRegression model and fit it on the training data
model = ...


# 4. Make predictions on the test set
# TODO: Use the trained model to predict on X_test
y_pred = ...


# 5. Visualize
plt.figure(figsize=(8, 5))
plt.scatter(X_test, y_test, color='blue', alpha=0.6, label='Actual')
plt.plot(X_test, y_pred, color='red', linewidth=2, label='Predicted')
plt.xlabel('Feature')
plt.ylabel('Target')
plt.title('Simple Linear Regression')
plt.legend()
plt.show()

# 6. Evaluate
# TODO: Print the R² score, MSE, the model coefficient (slope) and intercept

**Question 1:** What do the R² score and MSE tell us about the model's performance? What does the coefficient represent geometrically?

*Write your answer here:*

---
## Exercise 2: Multiple Linear Regression

Now we work with a real-world dataset: the **diabetes dataset** from scikit-learn. It has 10 numerical features and a quantitative target measuring disease progression.

### Steps:
1. Load the diabetes dataset.
2. Split into training and test sets.
3. Train a `LinearRegression` model.
4. Evaluate performance with R² and MSE.
5. Display the model coefficients to see which features are most important.

In [ ]:
from sklearn.datasets import load_diabetes

# 1. Load the dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_names}")

# 2. Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the model
# TODO: Create a LinearRegression model and fit it on the training data
model = ...


# 4. Evaluate
# TODO: Predict on X_test and print R² and MSE
y_pred = ...


# 5. Display coefficients
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': model.coef_})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print(f"\nModel coefficients:")
print(coef_df.to_string(index=False))

# Visualize coefficients
plt.figure(figsize=(10, 5))
plt.barh(coef_df['Feature'], coef_df['Coefficient'])
plt.xlabel('Coefficient value')
plt.title('Linear Regression Coefficients (Diabetes Dataset)')
plt.tight_layout()
plt.show()

**Question 2:** Which features have the largest impact on disease progression? Are any coefficients negative? What does a negative coefficient mean?

*Write your answer here:*

---
## Exercise 3: Regularization — Ridge and Lasso

Regularization adds a penalty to the loss function to reduce model complexity and prevent overfitting.

### Ridge Regression (L2 penalty)
$$J(\theta) = \sum_{i=1}^{m}(y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{n} \theta_j^2$$

Ridge shrinks coefficients towards zero but does not set them exactly to zero.

### Lasso Regression (L1 penalty)
$$J(\theta) = \sum_{i=1}^{m}(y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{n} |\theta_j|$$

Lasso can set some coefficients exactly to zero, performing **feature selection**.

### Steps:
1. Load the California Housing dataset.
2. Train LinearRegression, Ridge, and Lasso models.
3. Compare their R² scores.
4. Study the effect of the regularization parameter $\alpha$.
5. Compare model coefficients.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 1. Load the dataset
housing = fetch_california_housing()
X, y = housing.data, housing.target
feature_names = housing.feature_names

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_names}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 2 & 3. Train and compare models
# Note: we use StandardScaler because regularization is sensitive to feature scale
# TODO: Create 3 pipelines (LinearRegression, Ridge with alpha=1.0, Lasso with alpha=0.1),
#       each with a StandardScaler as the first step (use make_pipeline).
#       Train each on the training data, predict on the test data,
#       and print the R² score and MSE for each.

models = {
    'Linear Regression': ...,
    'Ridge (alpha=1.0)': ...,
    'Lasso (alpha=0.1)': ...,
}

In [ ]:
# 4. Study the effect of alpha
# TODO: For a range of alpha values (use np.logspace(-3, 3, 50)),
#       train Ridge and Lasso models and record their R² scores on the test set.
#       Then plot the R² scores as a function of alpha (use log scale on x-axis).

alphas = np.logspace(-3, 3, 50)
ridge_scores = []
lasso_scores = []

# TODO: Fill ridge_scores and lasso_scores


# Plot
plt.figure(figsize=(10, 5))
plt.plot(alphas, ridge_scores, label='Ridge', linewidth=2)
plt.plot(alphas, lasso_scores, label='Lasso', linewidth=2)
plt.xscale('log')
plt.xlabel('Alpha (regularization strength)')
plt.ylabel('R² Score')
plt.title('Effect of Regularization Parameter Alpha')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 5. Compare coefficients
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, (name, model) in zip(axes, models.items()):
    # The regression model is the second step of the pipeline (after StandardScaler)
    coefs = model[1].coef_
    ax.barh(feature_names, coefs)
    ax.set_title(name)
    ax.axvline(x=0, color='k', linestyle='--', alpha=0.5)

axes[0].set_ylabel('Feature')
fig.suptitle('Comparison of Model Coefficients', fontsize=14)
plt.tight_layout()
plt.show()

# Show which coefficients Lasso set to zero
lasso_model = models['Lasso (alpha=0.1)'][1]
zero_coefs = [f for f, c in zip(feature_names, lasso_model.coef_) if abs(c) < 1e-10]
print(f"Features set to zero by Lasso: {zero_coefs if zero_coefs else 'None (try a larger alpha)'}")

**Question 3:** 
- What happens to model performance when alpha is too large? Too small?
- What is the key difference between Ridge and Lasso in terms of feature selection?
- Why do we need to standardize features before applying regularization?

*Write your answer here:*

---
## Exercise 4: Logistic Regression for Classification

Logistic regression is used for **classification** tasks. Despite its name, it is not a regression model — it predicts the probability that an observation belongs to a given class.

### Steps:
1. Generate a 2D synthetic classification dataset.
2. Train a logistic regression model.
3. Visualize the decision boundary.
4. Compute the confusion matrix.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

# 1. Generate synthetic 2D data
X_class, y_class = make_classification(
    n_samples=300, n_features=2, n_redundant=0,
    n_informative=2, n_clusters_per_class=1, random_state=42
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.3, random_state=42
)

# 2. Train logistic regression
# TODO: Create a LogisticRegression model and fit it on the training data.
#       Print the model weights (coef_), bias (intercept_), and accuracy on the test set.
log_reg = ...

In [ ]:
# 3. Visualize the decision boundary
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap=plt.cm.RdYlBu, s=30)
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')

plt.figure(figsize=(8, 6))
plot_decision_boundary(log_reg, X_class, y_class, "Logistic Regression Decision Boundary")
plt.show()

In [ ]:
# 4. Confusion matrix
# TODO: Use the trained model to predict on the test set.
#       Compute the confusion matrix and display it as a pandas DataFrame.
#       Also print the classification report.

y_pred_c = ...

# Display as table
cm = confusion_matrix(y_test_c, y_pred_c)
cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"])
print("Confusion Matrix:")
print(cm_df)

print("\nClassification Report:")
print(classification_report(y_test_c, y_pred_c))

**Question 4:**
- What does each cell in the confusion matrix represent (TP, FP, TN, FN)?
- In what situations would you prefer to optimize recall over precision?

*Write your answer here:*

---
## Exercise 5: Poisson Regression — Modeling World Population Growth

### Background: Why not just use linear regression?

In standard linear regression, we model the target as a linear function of the features:

$$h_\theta(X) = X\theta = \theta_0 + \theta_1 x_1 + \cdots + \theta_n x_n$$

This works well when the target can take any real value. But for **population data**, we have two problems:
1. Population is always **positive** — linear regression could predict negative values.
2. Population growth is **exponential**, not linear — a straight line is a poor fit.

### The idea behind Poisson regression

Instead of predicting the target directly as a linear combination, we assume that the **logarithm** of the target is linear:

$$\log(h_\theta(X)) = X\theta$$

Equivalently, the prediction (hypothesis) is:

$$h_\theta(X) = \exp(X\theta) = \exp(\theta_0 + \theta_1 x_1 + \cdots + \theta_n x_n)$$

This guarantees that predictions are always **positive** and naturally captures **exponential** trends.

### Notation

- $X \in \mathbb{R}^{m \times (n+1)}$ is the feature matrix ($m$ samples, $n$ features + 1 bias column of ones)
- $\theta \in \mathbb{R}^{(n+1) \times 1}$ is the parameter vector (intercept $\theta_0$ + coefficients $\theta_1, \ldots, \theta_n$)
- $y \in \mathbb{R}^m$ is the target vector (here, population counts)

In our case, we have a single feature ($n = 1$): the year. So the model is simply:

$$h_\theta(\text{year}) = \exp(\theta_0 + \theta_1 \times \text{year})$$

The parameter $\theta_1$ controls the **growth rate**: a positive $\theta_1$ means exponential growth.

### Why is it called Poisson regression?

The [Poisson distribution](https://en.wikipedia.org/wiki/Poisson_distribution) is a probability distribution for count data ($0, 1, 2, \ldots$) with parameter $\lambda > 0$. In Poisson regression, we model the expected count as $\lambda = \exp(X\theta)$. The name comes from the statistical assumption on the distribution of the target, but in practice, the key idea is simply: **use an exponential link to ensure positive predictions**.

### Steps:
1. Load world population data from the CSV file.
2. Visualize the data (linear and log scale).
3. Fit a Poisson regression model using scikit-learn.
4. Evaluate the model and visualize predictions.
5. Make future population projections and discuss their limitations.

In [ ]:
# 1. Load world population data
pop_df = pd.read_csv("population-regions-with-projections.csv")

# Filter for world-level data
world = pop_df[pop_df["Entity"] == "World"].copy()
world = world.sort_values("Year")

print(f"World population data: {world.shape[0]} entries")
print(f"Year range: {world['Year'].min()} to {world['Year'].max()}")
world.tail(10)

In [ ]:
# 2. Visualize the data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(world["Year"], world["Population"], s=10, alpha=0.7)
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Population")
axes[0].set_title("World Population (Linear Scale)")

axes[1].scatter(world["Year"], world["Population"], s=10, alpha=0.7)
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Population")
axes[1].set_yscale("log")
axes[1].set_title("World Population (Log Scale)")

plt.tight_layout()
plt.show()

We restrict the data to years $\geq 1700$.

In [ ]:
from sklearn.linear_model import PoissonRegressor

# Restrict to years >= 1700
world_modern = world[world["Year"] >= 1700].copy()
Years = world_modern[["Year"]].values
Pop = world_modern["Population"].values

print(f"Data points (years >= 1700): {len(Pop)}")

# 3. Fit a Poisson regression model
# TODO: Create a PoissonRegressor (use solver="newton-cholesky" and alpha=0)
#       and fit it on Years and Pop.
#       Print the intercept (theta_0), coefficient (theta_1), and R² score.
poisson_model = ...

In [ ]:
# 4. Visualize the fit
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scale in zip(axes, ['linear', 'log']):
    ax.scatter(Years, Pop, s=15, alpha=0.7, label='Actual data')
    ax.plot(Years, poisson_model.predict(Years), 'r-', linewidth=2, label='Poisson fit')
    ax.set_xlabel("Year")
    ax.set_ylabel("Population")
    ax.set_yscale(scale)
    ax.set_title(f"Poisson Regression Fit ({scale} scale)")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 5. Make future projections
# TODO: Use poisson_model.predict() to estimate world population for 2030, 2050, 2100, and 2200.
#       Print the results in billions.

future_years = np.array([[2030], [2050], [2100], [2200]])

**Question 5:**
- What alternative name could you give to Poisson regression, based on its hypothesis function $h_\theta(X) = \exp(X\theta)$?
- Why might the model's far-future projections be inaccurate?

*Write your answer here:*